# VerdictLens EDA

Exploratory analysis of IN-Abs judgment and summary lengths, compression, and summary sentence position bias.

In [ ]:
from pathlib import Path
import random
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

with Path("configs/base.yaml").open(encoding="utf-8") as handle:
    config = yaml.safe_load(handle)
seed = int(config["seed"])
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
eda_dir = Path(config["paths"]["eda_output"])
eda_dir.mkdir(parents=True, exist_ok=True)
plt.rcParams["figure.dpi"] = 120
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
from src.data.loader import load_in_abs_dataset
dataset = load_in_abs_dataset("configs/base.yaml")
if dataset.empty:
    raise ValueError("The IN-Abs dataset is empty; add files under data/raw/.")
print(f"Loaded {len(dataset):,} documents")
dataset.head()

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config["models"]["led"])
def token_length(value: str) -> int:
    return len(tokenizer(value, add_special_tokens=True, truncation=False)["input_ids"])
dataset["judgment_tokens"] = dataset["text"].map(token_length)
dataset["summary_tokens"] = dataset["summary"].map(token_length)
dataset[["judgment_tokens", "summary_tokens"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99])

In [ ]:
thresholds = [1024, 4096, 16384]
exceedance = {threshold: float((dataset["judgment_tokens"] > threshold).mean() * 100) for threshold in thresholds}
for threshold, percentage in exceedance.items():
    print(f"Judgments over {threshold:,} tokens: {percentage:.2f}%")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(dataset["judgment_tokens"], bins=40, color="#174c63", alpha=0.85)
axes[0].set_title("Judgment token lengths")
axes[0].set_xlabel("LED tokens")
axes[0].set_ylabel("Documents")
axes[1].hist(dataset["summary_tokens"], bins=40, color="#d17a22", alpha=0.85)
axes[1].set_title("Summary token lengths")
axes[1].set_xlabel("LED tokens")
axes[1].set_ylabel("Documents")
fig.tight_layout()
fig.savefig(eda_dir / "token_length_distributions.png", bbox_inches="tight")
plt.show()

In [ ]:
dataset["compression_ratio"] = np.where(dataset["summary_tokens"] > 0, dataset["judgment_tokens"] / dataset["summary_tokens"], np.nan)
print(dataset["compression_ratio"].describe())
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(dataset["compression_ratio"].dropna(), bins=50, color="#2a7f62", alpha=0.85)
ax.set_title("Judgment-to-summary compression ratio")
ax.set_xlabel("Judgment tokens / summary tokens")
ax.set_ylabel("Documents")
fig.tight_layout()
fig.savefig(eda_dir / "compression_ratio.png", bbox_inches="tight")
plt.show()

In [ ]:
from nltk.tokenize import sent_tokenize
from rouge_score.rouge_scorer import RougeScorer
scorer = RougeScorer(["rougeL"], use_stemmer=True)
def split_sentences(text: str) -> list[str]:
    try:
        return [sentence.strip() for sentence in sent_tokenize(text) if sentence.strip()]
    except LookupError:
        return [sentence.strip() for sentence in re.split(r"(?<=[.!?])\s+", text) if sentence.strip()]
position_rows = []
for row in dataset[["doc_id", "text", "summary"]].itertuples(index=False):
    source_sentences = split_sentences(row.text)
    summary_sentences = split_sentences(row.summary)
    if not source_sentences or not summary_sentences:
        continue
    denominator = max(len(source_sentences) - 1, 1)
    for summary_sentence in summary_sentences:
        scores = [scorer.score(summary_sentence, source_sentence)["rougeL"].fmeasure for source_sentence in source_sentences]
        best_index = int(np.argmax(scores))
        position_rows.append({"doc_id": row.doc_id, "relative_position": best_index / denominator if len(source_sentences) > 1 else 0.5})
position_bias = pd.DataFrame(position_rows)
if position_bias.empty:
    raise ValueError("No sentence pairs were available for position-bias analysis.")
position_bias.head()

In [ ]:
print(position_bias["relative_position"].describe())
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(position_bias["relative_position"], bins=20, range=(0, 1), color="#8b3a62", alpha=0.85)
ax.set_title("Summary sentence position bias")
ax.set_xlabel("Relative position of best-matching source sentence")
ax.set_ylabel("Summary sentences")
ax.set_xlim(0, 1)
fig.tight_layout()
fig.savefig(eda_dir / "summary_sentence_position_bias.png", bbox_inches="tight")
plt.show()